In [1]:
import scipy as sp
from recpack.util import get_top_K_values, get_top_K_ranks
from metrics import calculate_ndcg, calculate_item_gini
import warnings
from scipy.sparse import SparseEfficiencyWarning
import pandas as pd

warnings.filterwarnings(
    "ignore",
    category=SparseEfficiencyWarning
)

Runs the preprocessing functions on the data and puts the processed data in variables. 

In [2]:
%%capture        
%run -n ./preprocessing.ipynb  
train_matrix, (val_fold_in, val_hold_out), (test_fold_in, test_hold_out), (train_users, val_users, test_users) = GetProcessedData()

In [3]:
def my_cosine_similarity(X: sp.sparse.csr_matrix) -> sp.sparse.csr_matrix:
    dot = X.T.dot(X)
    norm = sp.sparse.linalg.norm(X, ord=2, axis=0)
    normxnorm = np.outer(norm, norm)
    cosine = dot/normxnorm
    cosine = sp.sparse.csr_matrix(cosine)
    cosine.setdiag(0)
    cosine.eliminate_zeros()
    return cosine

The item-knn algorithm made for lecture 3, but I have added the knn scores to the recommendation dataframe for post processing methods and added a method to run the algorithm with given parameters and print the scores.

In [4]:
def matrix2df(X) -> pd.DataFrame:
    coo = sp.sparse.coo_array(X)
    return pd.DataFrame({
        "user_id": coo.row,
        "item_id": coo.col,
        "value": coo.data
    })

def scores2recommendations(
    scores: sp.sparse.csr_matrix, 
    X_test_in: sp.sparse.csr_matrix, 
    recommendation_count: int,
    prevent_history_recos = True
) -> pd.DataFrame:    
    # ensure you don't recommend fold-in items
    if prevent_history_recos:
        scores[(X_test_in > 0)] = 0
    # rank items
    ranks = get_top_K_ranks(scores, recommendation_count)
    # convert to a dataframe
    
    df_recos = matrix2df(ranks).rename(columns={"value": "rank"}).sort_values(["user_id", "rank"])
    # Add the KNN score to the recommendation dataframe
    scores_csr = scores.tocsr()
    df_recos["score"] = scores_csr[
        df_recos["user_id"].to_numpy(),
        df_recos["item_id"].to_numpy()
    ].A1
    return df_recos

def item_knn_scores(
    X_train: sp.sparse.csr_matrix, 
    X_test_in: sp.sparse.csr_matrix, 
    neighbor_count: int,
    damping_factor: int = 0,
    min_similarity: float = 0.0
) -> sp.sparse.csr_matrix:
    S = my_cosine_similarity(X_train)

    if damping_factor > 0:
        cooccurrence = (X_train.T @ X_train).tocsr().astype(float)
        cooccurrence.data = (cooccurrence.data / (cooccurrence.data + damping_factor))
        S = S.multiply(cooccurrence)

    S = get_top_K_values(S, neighbor_count)

    if min_similarity > 0.0:
        S[S < min_similarity] = 0
    
    return X_test_in * S

def run_knn_evaluation(X_train, X_test_in, X_test_out, neighbor_count, damping_factor=0, min_similarity=0.0):
    scores = item_knn_scores(X_train, X_test_in, neighbor_count, damping_factor, min_similarity)
    df_recos = scores2recommendations(scores, X_test_in, 20)
    ndcg = calculate_ndcg(df_recos, 20, matrix2df(X_test_out))
    gini = calculate_item_gini(df_recos, 20)
    return ndcg, gini

Some functions used to optimize the NDCG metric a bit on the base line algorithm. 

In [5]:
# Results: 7 tested best after several tests with smaller and smaller increments 
# around the best performing value from the previous test
NEIGHBOUR_COUNT = 7
def test_neighbor_counts():
    for neighbor_count in [6,7]:
        ndcg, gini = run_knn_evaluation(train_matrix, val_fold_in, val_hold_out, neighbor_count)
        print(f"Neighbor count: {neighbor_count}")
        print(f"NDCG@20: {ndcg:.5f}")
        print(f"Gini@20: {gini:.5f}")

# Results: Any sufficiently small threshold changes nothing, and at 0.08 the results get worse.
SIMILARITY_THRESHOLD = 0.0
def test_similarity_thresholds():
    for min_similarity in [0.01,0.03,0.05,0.07,0.09]:
        ndcg, gini = run_knn_evaluation(train_matrix, val_fold_in, val_hold_out, 7, 0, min_similarity)
        print(f"Min similarity: {min_similarity}")
        print(f"NDCG@20: {ndcg:.5f}")
        print(f"Gini@20: {gini:.5f}")

# Results: 7 tested as best
DAMPING_FACTOR = 7
def test_damping_factors():
    for damping_factor in [7,9]:
        ndcg, gini = run_knn_evaluation(train_matrix, val_fold_in, val_hold_out, 7, damping_factor, 0.0)
        print(f"Damping factor: {damping_factor}")
        print(f"NDCG@20: {ndcg:.5f}")
        print(f"Gini@20: {gini:.5f}")

# Final evaluation with best parameters
def optimal_baseline_evaluation():
    ndcg, gini = run_knn_evaluation(train_matrix, val_fold_in, val_hold_out, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
    print(f"Baseline: {NEIGHBOUR_COUNT} neighbors, {DAMPING_FACTOR} damping factor, {SIMILARITY_THRESHOLD} similarity threshold")
    print(f"NDCG@20: {ndcg:.5f}")
    print(f"Gini@20: {gini:.5f}")

A post processing method where we order the recommendations by popularity, then keep only the least popular items.
This is achieved by calculating the popularity of each item from the training data, then caluculating a popularity score by doing 1/log(popularity), and calculating a new score based on the weighted average of the knn score and the popularity score.
In order to introduce new recommendations and not just rerank them, we need to keep extra recommendations from the knn algorithm. 

In [6]:
def popularity_reranking(df_recos, X_train, alpha):
    item_popularity = np.asarray(X_train.sum(axis=0)).ravel()
    log_popularity = np.log1p(item_popularity)
    popularity = pd.DataFrame({
        "item_id": np.arange(len(item_popularity)),
        "popularity": log_popularity
    })
    
    result = df_recos.merge(popularity, on="item_id", how="left")

    result["adjusted_score"] = (1-alpha) * result["score"] + alpha * (1/result["popularity"])
    result = result.sort_values(["adjusted_score"], ascending=False)

    # Keep 20 least popular items per user
    result = (
        result.groupby("user_id", group_keys=False)
        .head(20)
        .copy()
    )
    
    # Reassign ranks after reranking
    result["rank"] = result.groupby("user_id").cumcount() + 1

    return result[["user_id", "item_id", "rank"]]

def run_knn_with_popularity_reranking(X_train, X_test_in, X_test_out, extra_recommendation_count, alpha):
    scores = item_knn_scores(X_train, X_test_in, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
    df_recos = scores2recommendations(scores, X_test_in, 20 + extra_recommendation_count)
    df_recos = popularity_reranking(df_recos, X_train, alpha)
    ndcg = calculate_ndcg(df_recos, 20, matrix2df(X_test_out))
    gini = calculate_item_gini(df_recos, 20)
    return ndcg, gini

def test_popularity_reranking():
    print("Popularity reranking: Alpha nDCG Gini")
    for alpha in [0.87, 0.89, 0.91, 0.93, 0.97]:
        ndcg, gini = run_knn_with_popularity_reranking(
            train_matrix,
            val_fold_in,
            val_hold_out,
            80,
            alpha
        )
        print(alpha, ndcg, gini)

Downsampling algo

In [7]:
def interaction_downsampling(X_train, max_interactions):
    downsampled_train_matrix = X_train.copy()

    rng = np.random.default_rng(42)
    X_csc = X_train.tocsc()

    rows = []
    cols = []
    values = []

    for item_id in range(X_csc.shape[1]):
        start = X_csc.indptr[item_id]
        end = X_csc.indptr[item_id + 1]

        item_rows = X_csc.indices[start:end]
        item_values = X_csc.data[start:end]

        if len(item_rows) > max_interactions:
            selected = rng.choice(
                len(item_rows),
                size=max_interactions,
                replace=False
            )
            item_rows = item_rows[selected]
            item_values = item_values[selected]

        rows.extend(item_rows)
        cols.extend([item_id] * len(item_rows))
        values.extend(item_values)

    downsampled_train_matrix = sp.sparse.coo_matrix(
        (values, (rows, cols)),
        shape=X_train.shape
    ).tocsr()

    return downsampled_train_matrix

def run_knn_with_downsampling(X_train, X_test_in, X_test_out, max_interactions):
    downsampled_train_matrix = interaction_downsampling(X_train, max_interactions)
    scores = item_knn_scores(downsampled_train_matrix, X_test_in, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
    df_recos = scores2recommendations(scores, X_test_in, 20)
    ndcg = calculate_ndcg(df_recos, 20, matrix2df(X_test_out))
    gini = calculate_item_gini(df_recos, 20)
    return ndcg, gini

def test_interaction_downsampling():
    print("Interaction downsampling: Max Interactions nDCG Gini")
    for max_interactions in [3000, 3500, 4000, 4500, 5000]:
        ndcg, gini = run_knn_with_downsampling(
            train_matrix,
            val_fold_in,
            val_hold_out,
            max_interactions
        )
        print(max_interactions, ndcg, gini)

Improved downsampling algorithm

In [8]:
def smarter_interaction_downsampling(X_train, max_interactions, alpha):
    downsampled_train_matrix = X_train.copy()

    rng = np.random.default_rng(42)
    X_csc = X_train.tocsc()

    rows = []
    cols = []
    values = []

    for item_id in range(X_csc.shape[1]):
        start = X_csc.indptr[item_id]
        end = X_csc.indptr[item_id + 1]

        item_rows = X_csc.indices[start:end]
        item_values = X_csc.data[start:end]

        n = len(item_rows)

        if n == 0:
            continue

        # P = min(1, (K / n)^alpha)
        keep_probability = min(
            1.0,
            (max_interactions / n) ** alpha
        )

        keep = rng.random(n) < keep_probability

        rows.extend(item_rows[keep])
        cols.extend([item_id] * np.sum(keep))
        values.extend(item_values[keep])

    downsampled_train_matrix = sp.sparse.coo_matrix(
        (values, (rows, cols)),
        shape=X_train.shape
    ).tocsr()

    assert downsampled_train_matrix.shape == X_train.shape, "Downsampled matrix shape does not match original shape"
    return downsampled_train_matrix

def run_knn_with_smart_downsampling(X_train, X_test_in, X_test_out, max_interactions, alpha):
    downsampled_train_matrix = smarter_interaction_downsampling(X_train, max_interactions, alpha)
    scores = item_knn_scores(downsampled_train_matrix, X_test_in, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
    df_recos = scores2recommendations(scores, X_test_in, 20)
    ndcg = calculate_ndcg(df_recos, 20, matrix2df(X_test_out))
    gini = calculate_item_gini(df_recos, 20)
    return ndcg, gini

def test_smart_downsampling():
    print("Smart downsampling: Alpha nDCG Gini")
    for alpha in [0.3, 0.35,0.4,0.45, 0.5]:
        ndcg, gini = run_knn_with_smart_downsampling(
            train_matrix,
            val_fold_in,
            val_hold_out,
            1000,
            alpha
        )
        print(alpha, ndcg, gini)
    

In [9]:
#optimal_baseline_evaluation()
#test_popularity_reranking()
#test_interaction_downsampling()
#test_smart_downsampling()

Generate files for codabench

In [ ]:
def save_submission(recs, user_mapping, item_mapping, path):
    recs = recs.sort_values(["user_id", "rank"])
    submission = recs[["user_id", "item_id"]].copy()
    submission["user_id"] = submission["user_id"].map(user_mapping)
    submission["item_id"] = submission["item_id"].map(item_mapping)
    submission = submission.astype(int)
    submission.to_csv(path, index=False)

def generate_submission(X_train, X_test, item_mapping, user_mapping_test, extra_recommendation_count, rerank_alpha, max_interactions, downsample_alpha, path):
    if (max_interactions is not None):
        downsampled_train_matrix = smarter_interaction_downsampling(X_train, max_interactions, downsample_alpha)
        scores = item_knn_scores(downsampled_train_matrix, X_test, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
        df_recos = scores2recommendations(scores, X_test, 20)
    elif (rerank_alpha is not None):
        scores = item_knn_scores(X_train, X_test, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
        df_recos = scores2recommendations(scores, X_test, 20 + extra_recommendation_count)
        df_recos = popularity_reranking(df_recos, X_test, rerank_alpha)
    else:
        scores = item_knn_scores(X_train, X_test, NEIGHBOUR_COUNT, DAMPING_FACTOR, SIMILARITY_THRESHOLD)
        df_recos = scores2recommendations(scores, X_test, 20)
    save_submission(df_recos, user_mapping_test, item_mapping, path)

test_matrix, train_matrix, item_mapping, user_mapping_test, user_mapping = GetCodaBenchTestData()

print("Baseline")
generate_submission(
    train_matrix,
    test_matrix,
    item_mapping,
    user_mapping_test,
    extra_recommendation_count=None,
    rerank_alpha=None,
    max_interactions=None,
    downsample_alpha=None,
    path="../codabench_submissions/baseline.csv"
)
print("Reranking")
generate_submission(
    train_matrix,
    test_matrix,
    item_mapping,
    user_mapping_test,
    extra_recommendation_count=80,
    rerank_alpha=0.97,
    max_interactions=None,
    downsample_alpha=None,
    path="../codabench_submissions/reranking.csv"
)
print("Downsampling")
generate_submission(
    train_matrix,
    test_matrix,
    item_mapping,
    user_mapping_test,
    extra_recommendation_count=None,
    rerank_alpha=None,
    max_interactions=1000,
    downsample_alpha=0.5,
    path="../codabench_submissions/downsampling.csv"
)
        

Baseline
Reranking
Downsampling
